# LightGBM accidentologie - 3 runs d'optimisation

Objectif:
- entrainer 3 runs LightGBM avec variation d'hyperparametres
- comparer les performances et selectionner le meilleur
- exporter modeles, predictions et metadonnees
- logger dans MLflow (meme experience que CatBoost, RF, XGBoost, LogReg)

In [1]:
from __future__ import annotations

import gc
import json
from datetime import UTC, datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    StratifiedKFold,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder


def find_project_root(marker: str = "out") -> Path:
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / marker).exists():
            return p
    return Path.cwd()


ROOT = find_project_root("out")
DATA_PATH = ROOT / "out" / "accidents_model_ready_kept_with_time_bucket.csv"
ARTIFACT_DIR = ROOT / "out" / "lgbm_experiments"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "grave"
SEED = 42
CV_SPLITS = 3
N_ITER = 10  # reduit de 20 a 10 pour limiter la RAM (16 Go)
LGBM_N_JOBS = 2  # LightGBM interne : 2 threads (pas tous les cores)
SEARCH_N_JOBS = 1  # pas de parallelisme sur le search (1 modele a la fois)
MAX_TRAIN_SAMPLES = 60_000  # sous-echantillon pour securiser la RAM
THRESHOLD = 0.5

# Memes 15 features que les autres notebooks pour comparaison equitable
PRODUCT15_V2 = [
    "dep",
    "lum",
    "atm",
    "catr",
    "agg",
    "int",
    "circ",
    "col",
    "vma_bucket",
    "catv_family_4",
    "manv_mode",
    "driver_age_bucket",
    "choc_mode",
    "driver_trajet_family",
    "time_bucket",
]

print("ROOT:", ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print(
    "Config:",
    {
        "cv_splits": CV_SPLITS,
        "n_iter": N_ITER,
        "lgbm_n_jobs": LGBM_N_JOBS,
        "search_n_jobs": SEARCH_N_JOBS,
        "max_train_samples": MAX_TRAIN_SAMPLES,
    },
)

ROOT: /home/maxime/simplonalternance/alternance-CICDprediction
DATA_PATH: /home/maxime/simplonalternance/alternance-CICDprediction/out/accidents_model_ready_kept_with_time_bucket.csv
ARTIFACT_DIR: /home/maxime/simplonalternance/alternance-CICDprediction/out/lgbm_experiments
Config: {'cv_splits': 3, 'n_iter': 10, 'lgbm_n_jobs': 2, 'search_n_jobs': 1, 'max_train_samples': 60000}


In [2]:
assert DATA_PATH.exists(), f"Fichier introuvable: {DATA_PATH}"

read_cols = PRODUCT15_V2 + [TARGET]
df = pd.read_csv(
    DATA_PATH,
    sep=";",
    usecols=read_cols,
    dtype=dict.fromkeys(PRODUCT15_V2, "category"),
    low_memory=False,
)

assert TARGET in df.columns, f"Colonne cible absente: {TARGET}"
missing_feats = [c for c in PRODUCT15_V2 if c not in df.columns]
assert not missing_feats, f"Colonnes manquantes dans le CSV: {missing_feats}"

df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=[TARGET]).copy()
df[TARGET] = df[TARGET].astype(int)

X = df[PRODUCT15_V2].copy()
y = df[TARGET].copy()

print("Shape:", X.shape)
print("Taux grave=1:", round(float(y.mean()), 4))
print("Features utilisees:", list(X.columns))

Shape: (164526, 15)
Taux grave=1: 0.3608
Features utilisees: ['dep', 'lum', 'atm', 'catr', 'agg', 'int', 'circ', 'col', 'vma_bucket', 'catv_family_4', 'manv_mode', 'driver_age_bucket', 'choc_mode', 'driver_trajet_family', 'time_bucket']


## Preprocessing

- Split train/val/test stratifie (60/20/20)
- **val** sert a optimiser le seuil, **test** sert a evaluer les metriques finales
- LightGBM gere les categories nativement via OrdinalEncoder
- Les 15 features sont toutes categorielles

In [3]:
# Split 60/20/20 : train / val (seuil) / test (evaluation finale)
X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.25,
    random_state=SEED,
    stratify=y_temp,
)

# Toutes les features product15_v2 sont categorielles
cat_cols = PRODUCT15_V2[:]
num_cols = []

cat_pipe = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1),
        ),
    ]
)

preprocess = ColumnTransformer(
    transformers=[("cat", cat_pipe, cat_cols)],
    remainder="drop",
)

pos = int((y_train == 1).sum())
neg = int((y_train == 0).sum())
scale_pos_weight_base = float(neg / max(pos, 1))

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)
print("Categorielles:", len(cat_cols))
print(
    "Class balance train -> pos:",
    pos,
    "neg:",
    neg,
    "scale_pos_weight:",
    round(scale_pos_weight_base, 3),
)

Train: (98715, 15) | Val: (32905, 15) | Test: (32906, 15)
Categorielles: 15
Class balance train -> pos: 35612 neg: 63103 scale_pos_weight: 1.772


## Trois runs d'optimisation

Runs prevus:
1. `lgbm_auc_opt` optimise `roc_auc`
2. `lgbm_f1_opt` optimise `f1`
3. `lgbm_recall_opt` optimise `recall`

In [4]:
def evaluate_binary(
    y_true: pd.Series, proba: np.ndarray, threshold: float = THRESHOLD
) -> dict[str, float]:
    pred = (proba >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": float(accuracy_score(y_true, pred)),
        "precision": float(precision_score(y_true, pred, zero_division=0)),
        "recall": float(recall_score(y_true, pred, zero_division=0)),
        "f1": float(f1_score(y_true, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, proba)),
        "pr_auc": float(average_precision_score(y_true, proba)),
    }


def best_threshold_by_f1(
    y_true: pd.Series, proba: np.ndarray
) -> tuple[float, dict[str, float]]:
    thresholds = np.linspace(0.05, 0.95, 91)
    rows_thr = [evaluate_binary(y_true, proba, float(t)) for t in thresholds]
    df_thr = pd.DataFrame(rows_thr)
    idx = int(df_thr["f1"].idxmax())
    best = df_thr.loc[idx].to_dict()
    return float(best["threshold"]), {k: float(v) for k, v in best.items()}


def make_search(scoring: str, param_dist: dict) -> RandomizedSearchCV:
    model = LGBMClassifier(
        objective="binary",
        metric="auc",
        random_state=SEED,
        n_jobs=LGBM_N_JOBS,  # 2 threads internes (pas -1)
        verbosity=-1,
    )

    pipe = Pipeline(steps=[("prep", preprocess), ("model", model)])
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=SEED)

    return RandomizedSearchCV(
        estimator=pipe,
        param_distributions=param_dist,
        n_iter=N_ITER,
        scoring=scoring,
        n_jobs=SEARCH_N_JOBS,  # 1 seul fit a la fois (pas -1)
        pre_dispatch=SEARCH_N_JOBS,
        cv=cv,
        random_state=SEED,
        verbose=1,
        refit=True,
    )


spw_mid = round(scale_pos_weight_base, 3)
spw_light = round(scale_pos_weight_base * 0.75, 3)
spw_high = round(scale_pos_weight_base * 1.5, 3)
spw_very_high = round(scale_pos_weight_base * 2.5, 3)

# ---- Grilles securisees pour 16 Go RAM ----
# n_estimators max 800 (au lieu de 1700)
# num_leaves max 127 (au lieu de 255)
# max_depth borne (pas de -1 illimite)

PARAM_AUC = {
    "model__n_estimators": [200, 400, 600, 800],
    "model__max_depth": [6, 8, 12],
    "model__learning_rate": [0.03, 0.05, 0.08, 0.12],
    "model__num_leaves": [31, 63, 127],
    "model__subsample": [0.7, 0.85],
    "model__colsample_bytree": [0.6, 0.8],
    "model__min_child_samples": [10, 20, 50],
    "model__reg_alpha": [0.0, 0.1, 0.5],
    "model__reg_lambda": [0.0, 1.0, 5.0],
    "model__scale_pos_weight": [spw_light, spw_mid, spw_high],
}

PARAM_F1 = {
    "model__n_estimators": [200, 400, 600, 800],
    "model__max_depth": [5, 8, 12],
    "model__learning_rate": [0.02, 0.03, 0.05, 0.08],
    "model__num_leaves": [31, 63, 127],
    "model__subsample": [0.75, 0.9],
    "model__colsample_bytree": [0.6, 0.8],
    "model__min_child_samples": [10, 20, 40],
    "model__reg_alpha": [0.0, 0.05, 0.2],
    "model__reg_lambda": [0.0, 1.0, 3.0],
    "model__scale_pos_weight": [spw_mid, spw_high],
}

PARAM_RECALL = {
    "model__n_estimators": [200, 400, 600, 800],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.02, 0.03, 0.05],
    "model__num_leaves": [31, 63, 127],
    "model__subsample": [0.75, 0.9],
    "model__colsample_bytree": [0.6, 0.8],
    "model__min_child_samples": [10, 20, 30],
    "model__reg_alpha": [0.0, 0.1, 0.4],
    "model__reg_lambda": [0.0, 1.0, 5.0],
    "model__scale_pos_weight": [spw_mid, spw_high, spw_very_high],
}

EXPERIMENTS = [
    ("lgbm_auc_opt", "roc_auc", PARAM_AUC),
    ("lgbm_f1_opt", "f1", PARAM_F1),
    ("lgbm_recall_opt", "recall", PARAM_RECALL),
]

In [5]:
trained = {}
rows = []

# Sous-echantillonnage pour securiser la RAM
if MAX_TRAIN_SAMPLES is not None and len(X_train) > MAX_TRAIN_SAMPLES:
    X_search, _, y_search, _ = train_test_split(
        X_train,
        y_train,
        train_size=MAX_TRAIN_SAMPLES,
        stratify=y_train,
        random_state=SEED,
    )
else:
    X_search, y_search = X_train, y_train

print(f"Search dataset: {X_search.shape}")

for run_name, scoring, param_dist in EXPERIMENTS:
    print(f"\n=== {run_name} | scoring={scoring} ===")

    search = make_search(scoring=scoring, param_dist=param_dist)
    search.fit(X_search, y_search)

    best_model = search.best_estimator_

    # Seuil optimise sur val (pas sur test)
    proba_val = best_model.predict_proba(X_val)[:, 1]
    best_thr, _ = best_threshold_by_f1(y_val, proba_val)

    # Metriques finales sur test (jamais vu par le modele ni le seuil)
    proba_test = best_model.predict_proba(X_test)[:, 1]
    metrics_05 = evaluate_binary(y_test, proba_test, threshold=THRESHOLD)
    metrics_best = evaluate_binary(y_test, proba_test, threshold=best_thr)

    trained[run_name] = {
        "search": search,
        "model": best_model,
        "proba_test": proba_test,
        "metrics_05": metrics_05,
        "best_threshold": best_thr,
        "metrics_best": metrics_best,
    }

    rows.append(
        {
            "run_name": run_name,
            "optimized_for": scoring,
            "cv_best_score": float(search.best_score_),
            "threshold_05": float(THRESHOLD),
            "accuracy_05": metrics_05["accuracy"],
            "precision_05": metrics_05["precision"],
            "recall_05": metrics_05["recall"],
            "f1_05": metrics_05["f1"],
            "roc_auc_05": metrics_05["roc_auc"],
            "pr_auc_05": metrics_05["pr_auc"],
            "best_threshold_f1": best_thr,
            "precision_best": metrics_best["precision"],
            "recall_best": metrics_best["recall"],
            "f1_best": metrics_best["f1"],
            "best_params": search.best_params_,
        }
    )

    # Liberer la memoire entre les experiences
    del search
    gc.collect()

results_df = (
    pd.DataFrame(rows)
    .sort_values(["f1_best", "roc_auc_05"], ascending=False)
    .reset_index(drop=True)
)
results_df[
    [
        "run_name",
        "optimized_for",
        "cv_best_score",
        "f1_05",
        "roc_auc_05",
        "f1_best",
        "best_threshold_f1",
    ]
]

Search dataset: (60000, 15)

=== lgbm_auc_opt | scoring=roc_auc ===
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/a


=== lgbm_f1_opt | scoring=f1 ===
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/a


=== lgbm_recall_opt | scoring=recall ===
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/alternance-CICDprediction/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/maxime/simplonalternance/a

,run_name,optimized_for,cv_best_score,f1_05,roc_auc_05,f1_best,best_threshold_f1
0,lgbm_f1_opt,f1,0.673009,0.679683,0.814785,0.679808,0.48
1,lgbm_auc_opt,roc_auc,0.809258,0.673301,0.814812,0.679462,0.56
2,lgbm_recall_opt,recall,0.913976,0.630396,0.806375,0.669786,0.69


In [6]:
registry_rows = []

for row in rows:
    run_name = row["run_name"]
    payload = trained[run_name]

    model = payload["model"]
    proba_test = payload["proba_test"]
    best_thr = payload["best_threshold"]

    model_path = ARTIFACT_DIR / f"{run_name}.joblib"
    pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
    meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

    joblib.dump(model, model_path)

    pred_df = pd.DataFrame(
        {
            "y_true": y_test.to_numpy(),
            "proba": proba_test,
            "pred_05": (proba_test >= THRESHOLD).astype(int),
            "pred_best_f1": (proba_test >= best_thr).astype(int),
        }
    )
    pred_df.to_csv(pred_path, index=False)

    meta = {
        "run_name": run_name,
        "optimized_for": row["optimized_for"],
        "dataset": str(DATA_PATH),
        "target": TARGET,
        "seed": SEED,
        "cv_splits": CV_SPLITS,
        "n_iter": N_ITER,
        "cv_best_score": row["cv_best_score"],
        "threshold_05": THRESHOLD,
        "best_threshold_f1": best_thr,
        "metrics_05": payload["metrics_05"],
        "metrics_best_f1": payload["metrics_best"],
        "best_params": payload["search"].best_params_,
        "model_path": str(model_path),
        "predictions_path": str(pred_path),
        "created_at_utc": datetime.now(UTC).isoformat(),
    }

    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, ensure_ascii=False, indent=2)

    registry_rows.append(
        {
            "run_name": run_name,
            "model_path": str(model_path),
            "predictions_path": str(pred_path),
            "meta_path": str(meta_path),
        }
    )

registry_df = pd.DataFrame(registry_rows)
summary_df = results_df.merge(registry_df, on="run_name", how="left")
summary_df

,run_name,optimized_for,cv_best_score,threshold_05,accuracy_05,precision_05,recall_05,f1_05,roc_auc_05,pr_auc_05,best_threshold_f1,precision_best,recall_best,f1_best,best_params,model_path,predictions_path,meta_path
0,lgbm_f1_opt,f1,0.673009,0.5,0.744727,0.620915,0.750737,0.679683,0.814785,0.702720,0.48,0.609515,0.768427,0.679808,"{'model__subsample': 0.75, 'model__scale_pos_w...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
1,lgbm_auc_opt,roc_auc,0.809258,0.5,0.711694,0.569432,0.823520,0.673301,0.814812,0.702010,0.56,0.603771,0.776851,0.679462,"{'model__subsample': 0.7, 'model__scale_pos_we...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...
2,lgbm_recall_opt,recall,0.913976,0.5,0.610223,0.479085,0.921405,0.630396,0.806375,0.687605,0.69,0.595155,0.765816,0.669786,"{'model__subsample': 0.9, 'model__scale_pos_we...",/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...,/home/maxime/simplonalternance/alternance-CICD...


In [7]:
best_idx = int(results_df["f1_best"].idxmax())
best_run = results_df.loc[best_idx, "run_name"]
best_params = results_df.loc[best_idx, "best_params"]

print("Best run (F1 best threshold):", best_run)
print("Best params:")
print(best_params)

Best run (F1 best threshold): lgbm_f1_opt
Best params:
{'model__subsample': 0.75, 'model__scale_pos_weight': 1.772, 'model__reg_lambda': 1.0, 'model__reg_alpha': 0.05, 'model__num_leaves': 63, 'model__n_estimators': 200, 'model__min_child_samples': 40, 'model__max_depth': 12, 'model__learning_rate': 0.05, 'model__colsample_bytree': 0.6}


## Integration MLflow

Log les 3 runs LightGBM dans l'experience `accidentologie_model_benchmark`.
Le meilleur run (par F1) est enregistre dans le Model Registry.

In [ ]:
ENABLE_MLFLOW = True
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT = "accidentologie_model_benchmark"
ENABLE_MODEL_REGISTRY = True
REGISTERED_MODEL_NAME = "briefml-lgbm-product15-v2-time-bucket"

if ENABLE_MLFLOW:
    import mlflow
    import mlflow.sklearn
    import numpy as np
    from mlflow.tracking import MlflowClient

    def _to_params(d):
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if isinstance(
                    v, int | float | str | bool | np.integer | np.floating | np.bool_
                ):
                    out[k] = v.item() if hasattr(v, "item") else v
        return out

    def _normalize_metrics(d, prefix="valid_"):
        keys = {
            "accuracy",
            "precision",
            "recall",
            "f1",
            "roc_auc",
            "pr_auc",
            "threshold",
        }
        out = {}
        if isinstance(d, dict):
            for k, v in d.items():
                if k in keys and isinstance(v, int | float | np.integer | np.floating):
                    out[f"{prefix}{k}"] = float(v)
        return out

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)
    client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

    # On enregistre le meilleur run (par f1_best) dans le registry
    best_run_name = results_df.loc[results_df["f1_best"].idxmax(), "run_name"]

    for row in rows:
        run_name = row["run_name"]
        payload = trained.get(run_name, {})
        search = payload.get("search")
        model = payload.get("model")

        if search is None or model is None:
            print(f"[mlflow] skip {run_name}: search/model manquant")
            continue

        metrics_05 = _normalize_metrics(payload.get("metrics_05"), prefix="valid_")
        metrics_best = _normalize_metrics(
            payload.get("metrics_best"), prefix="valid_bestf1_"
        )

        model_path = ARTIFACT_DIR / f"{run_name}.joblib"
        pred_path = ARTIFACT_DIR / f"{run_name}_predictions.csv"
        meta_path = ARTIFACT_DIR / f"{run_name}_meta.json"

        # Enregistrer dans le registry uniquement le meilleur run
        register_name = (
            REGISTERED_MODEL_NAME
            if (ENABLE_MODEL_REGISTRY and run_name == best_run_name)
            else None
        )

        with mlflow.start_run(run_name=run_name):
            mlflow.set_tags(
                {
                    "notebook": "16_lightgbm_optimization.ipynb",
                    "model_family": "lightgbm",
                    "model_flavor": "sklearn",
                    "tag": "accidentologie",
                    "optimized_for": str(row.get("optimized_for", "unknown")),
                }
            )
            if register_name:
                mlflow.set_tag("registry_enabled", "true")

            mlflow.log_param("seed", int(SEED))
            mlflow.log_param("cv_splits", int(CV_SPLITS))
            mlflow.log_param("n_iter", int(N_ITER))
            mlflow.log_param("target", TARGET)
            mlflow.log_param(
                "cv_primary_metric", str(row.get("optimized_for", "unknown"))
            )
            mlflow.log_params(_to_params(search.best_params_))
            mlflow.log_metric("cv_primary_score", float(search.best_score_))

            if metrics_05:
                mlflow.log_metrics(metrics_05)
            if metrics_best:
                mlflow.log_metrics(metrics_best)

            model_info = mlflow.sklearn.log_model(
                model,
                artifact_path="model",
                registered_model_name=register_name,
            )

            for p, art in [
                (model_path, "models"),
                (pred_path, "predictions"),
                (meta_path, "metadata"),
            ]:
                if p.exists():
                    mlflow.log_artifact(str(p), artifact_path=art)

            # --- Tags de performance sur le Model Registry ---
            if register_name and getattr(model_info, "registered_model_version", None):
                version = model_info.registered_model_version
                perf_tags = {
                    "f1_best": f"{payload['metrics_best']['f1']:.4f}",
                    "roc_auc": f"{payload['metrics_05']['roc_auc']:.4f}",
                    "recall_best": f"{payload['metrics_best']['recall']:.4f}",
                    "precision_best": f"{payload['metrics_best']['precision']:.4f}",
                    "best_threshold": f"{payload['best_threshold']:.2f}",
                    "optimized_for": str(row.get("optimized_for", "unknown")),
                }
                for tag_key, tag_val in perf_tags.items():
                    client.set_model_version_tag(
                        register_name, version, tag_key, tag_val
                    )
                print(f"[mlflow] model registered: {register_name} v{version}")
                print(f"[mlflow] registry tags: {perf_tags}")

            print(f"[mlflow] logged: {run_name}")
else:
    print("MLflow desactive. Passe ENABLE_MLFLOW=True pour logger les runs LightGBM.")